# LangChain Expression Language (LCEL) 学习 Demo

## 📚 简介

LangChain Expression Language (LCEL) 是 LangChain 提供的一种声明式编程方式，用于轻松组合链式调用。

### 核心优势：
- 🔗 **链式组合**：使用管道操作符 `|` 轻松连接多个组件
- 🚀 **流式支持**：内置流式输出功能
- 🔄 **批处理**：高效处理多个输入
- ⚡ **并行执行**：自动并行化独立操作
- 🎯 **统一接口**：所有组件都实现 Runnable 接口

### 本 Demo 涵盖内容：
1. 环境设置和基础配置
2. 基础链式调用
3. Prompt + LLM + Output Parser 组合
4. 流式处理
5. 批处理
6. 并行处理
7. 条件分支和路由
8. RunnableLambda 自定义处理
9. RunnablePassthrough 数据传递
10. 完整实战案例

## 🔧 1. 环境设置

### 安装依赖包

In [ ]:
# 安装所需依赖（如果还未安装）
# !pip install pydantic==2.10.3
# !pip install langchain==0.3.15
# !pip install langchain-core==0.3.28
# !pip install langchain-community==0.3.14
# !pip install dashscope==1.20.11
# !pip install python-dotenv

### 导入库和配置 API Key

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel, RunnableBranch
from langchain_core.messages import HumanMessage, SystemMessage

# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 初始化大模型
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.7,
)

print("✅ 环境配置完成！")

## 🔗 2. 基础链式调用

### 概念说明

LCEL 的核心是使用 `|` 操作符将多个组件连接成一个链。每个组件都实现了 `Runnable` 接口，具有以下方法：
- `invoke()`: 单个输入调用
- `batch()`: 批量输入调用
- `stream()`: 流式输出
- `ainvoke()`, `abatch()`, `astream()`: 对应的异步版本

### 案例 1: 最简单的链 - Prompt + LLM

In [ ]:
# 创建一个简单的提示词模板
prompt = ChatPromptTemplate.from_template("给我讲一个关于{topic}的笑话")

# 使用 | 操作符连接 prompt 和 llm
simple_chain = prompt | llm

# 调用链
result = simple_chain.invoke({"topic": "程序员"})
print("📝 输出类型:", type(result))
print("\n💬 回答:", result.content)

### 案例 2: 完整链 - Prompt + LLM + Output Parser

Output Parser 用于将 LLM 的输出转换为我们需要的格式。

In [ ]:
# 创建输出解析器
output_parser = StrOutputParser()

# 完整的链：Prompt -> LLM -> Parser
complete_chain = prompt | llm | output_parser

# 调用链
result = complete_chain.invoke({"topic": "数据科学家"})
print("📝 输出类型:", type(result))
print("\n💬 回答:", result)

## 🌊 3. 流式处理 (Streaming)

### 概念说明

流式处理允许我们在 LLM 生成内容时实时获取输出，而不是等待全部内容生成完成。这对于改善用户体验非常重要。

### 案例: 流式输出故事

In [ ]:
# 创建一个讲故事的链
story_prompt = ChatPromptTemplate.from_template(
    "用50字以内讲一个关于{subject}的小故事"
)
story_chain = story_prompt | llm | StrOutputParser()

# 使用 stream() 方法进行流式输出
print("🌊 流式输出开始:\n")
for chunk in story_chain.stream({"subject": "勇敢的小兔子"}):  
    print(chunk, end="", flush=True)

print("\n\n✅ 流式输出完成！")

## 📦 4. 批处理 (Batch Processing)

### 概念说明

批处理允许我们一次性处理多个输入，LangChain 会自动优化并行处理，提高效率。

### 案例: 批量生成标题

In [ ]:
# 创建标题生成链
title_prompt = ChatPromptTemplate.from_template(
    "为一篇关于{topic}的文章生成一个吸引人的标题（10字以内）"
)
title_chain = title_prompt | llm | StrOutputParser()

# 准备多个输入
topics = [
    {"topic": "人工智能"},
    {"topic": "区块链技术"},
    {"topic": "量子计算"},
]

# 使用 batch() 批量处理
print("📦 批量生成标题:\n")
results = title_chain.batch(topics)

for topic, title in zip(topics, results):
    print(f"主题: {topic['topic']:10s} -> 标题: {title}")

## ⚡ 5. 并行处理 (Parallel Execution)

### 概念说明

`RunnableParallel` (也可以用字典语法) 允许我们并行执行多个独立的链，然后将结果合并。

### 案例: 同时生成标题、摘要和关键词

In [ ]:
# 定义三个不同的链
title_chain = (
    ChatPromptTemplate.from_template("为{content}生成一个标题（10字以内）")
    | llm
    | StrOutputParser()
)

summary_chain = (
    ChatPromptTemplate.from_template("用一句话总结：{content}")
    | llm
    | StrOutputParser()
)

keywords_chain = (
    ChatPromptTemplate.from_template("提取{content}的3个关键词，用逗号分隔")
    | llm
    | StrOutputParser()
)

# 使用 RunnableParallel 并行执行（也可以直接用字典）
parallel_chain = RunnableParallel(
    title=title_chain,
    summary=summary_chain,
    keywords=keywords_chain
)

# 或者使用字典语法（更简洁）
# parallel_chain = {
#     "title": title_chain,
#     "summary": summary_chain,
#     "keywords": keywords_chain
# }

# 执行并行链
content = {"content": "机器学习是人工智能的一个分支，它使计算机能够从数据中学习并改进性能"}
result = parallel_chain.invoke(content)

print("⚡ 并行处理结果:\n")
print(f"📌 标题: {result['title']}")
print(f"📝 摘要: {result['summary']}")
print(f"🏷️  关键词: {result['keywords']}")

## 🔀 6. 条件分支 (Conditional Routing)

### 概念说明

`RunnableBranch` 允许我们根据条件选择不同的执行路径。这在需要根据输入动态选择处理逻辑时非常有用。

### 案例: 根据文本长度选择不同的处理方式

In [ ]:
# 定义不同的处理链
short_text_chain = (
    ChatPromptTemplate.from_template("这是一段短文本：{text}。请扩展它，使其更详细。")
    | llm
    | StrOutputParser()
)

long_text_chain = (
    ChatPromptTemplate.from_template("这是一段长文本：{text}。请总结其核心要点。")
    | llm
    | StrOutputParser()
)

# 使用 RunnableBranch 创建条件分支
branch_chain = RunnableBranch(
    # (条件函数, 执行的链)
    (lambda x: len(x["text"]) < 20, short_text_chain),
    (lambda x: len(x["text"]) >= 20, long_text_chain),
    # 默认链
    short_text_chain
)

# 测试短文本
print("🔀 测试条件分支:\n")
print("--- 短文本测试 ---")
short_result = branch_chain.invoke({"text": "Python很棒"})
print(f"输入: Python很棒 (长度: 7)")
print(f"输出: {short_result}\n")

# 测试长文本
print("--- 长文本测试 ---")
long_result = branch_chain.invoke({
    "text": "人工智能正在改变世界，机器学习和深度学习技术在各个领域都有广泛应用"
})
print(f"输入: 人工智能正在改变世界... (长度: 34)")
print(f"输出: {long_result}")

## 🎯 7. RunnableLambda - 自定义处理函数

### 概念说明

`RunnableLambda` 允许我们将普通的 Python 函数转换为 Runnable，方便在链中插入自定义逻辑。

### 案例: 文本预处理和后处理

In [ ]:
# 定义预处理函数
def preprocess(input_dict):
    """清理和规范化输入文本"""
    text = input_dict["text"]
    # 去除多余空格，转换为小写
    cleaned = " ".join(text.split()).strip()
    print(f"🔧 预处理: '{text}' -> '{cleaned}'")
    return {"text": cleaned}

# 定义后处理函数
def postprocess(output):
    """格式化输出"""
    formatted = f"\n{'='*50}\n{output}\n{'='*50}"
    return formatted

# 创建包含自定义函数的链
custom_chain = (
    RunnableLambda(preprocess)  # 预处理
    | ChatPromptTemplate.from_template("将以下文本翻译成英文：{text}")
    | llm
    | StrOutputParser()
    | RunnableLambda(postprocess)  # 后处理
)

# 测试
result = custom_chain.invoke({"text": "  你好，   世界！  "})
print("\n🎯 最终结果:")
print(result)

## 🔄 8. RunnablePassthrough - 数据传递

### 概念说明

`RunnablePassthrough` 用于在链中传递数据，可以:
- 保持原始输入不变
- 分配新的键值对
- 在并行处理中保留上下文

### 案例: 保留原始问题并生成答案

In [ ]:
# 创建 QA 链，同时保留原始问题
qa_chain = (
    # 使用 RunnablePassthrough.assign 添加新的键值对
    RunnablePassthrough.assign(
        answer=(
            ChatPromptTemplate.from_template("请回答以下问题：{question}")
            | llm
            | StrOutputParser()
        )
    )
)

# 测试
result = qa_chain.invoke({"question": "什么是LCEL?"})

print("🔄 RunnablePassthrough 示例:\n")
print(f"📌 原始问题: {result['question']}")
print(f"💡 生成答案: {result['answer']}")

## 🎨 9. 复杂组合 - Passthrough 与并行处理

### 概念说明

将 `RunnablePassthrough` 与并行处理结合，可以创建复杂的数据流。

### 案例: 问题分析系统

In [ ]:
# 创建复杂的问题分析链
analysis_chain = (
    # 保留原始输入并添加多个分析维度
    RunnablePassthrough.assign(
        difficulty=(
            ChatPromptTemplate.from_template(
                "评估问题难度（简单/中等/困难）：{question}"
            )
            | llm
            | StrOutputParser()
        ),
        category=(
            ChatPromptTemplate.from_template(
                "这个问题属于什么类别（技术/商业/科学/其他）：{question}"
            )
            | llm
            | StrOutputParser()
        ),
        answer=(
            ChatPromptTemplate.from_template(
                "用一句话回答：{question}"
            )
            | llm
            | StrOutputParser()
        )
    )
)

# 测试
result = analysis_chain.invoke({
    "question": "如何优化深度学习模型的训练速度？"
})

print("🎨 复杂问题分析结果:\n")
print(f"❓ 问题: {result['question']}")
print(f"📊 难度: {result['difficulty']}")
print(f"🏷️  类别: {result['category']}")
print(f"💡 答案: {result['answer']}")

## 🔍 10. JSON 输出解析

### 概念说明

`JsonOutputParser` 可以将 LLM 输出解析为结构化的 JSON 数据。

### 案例: 结构化信息提取

In [ ]:
from pydantic import BaseModel, Field

# 定义输出结构
class PersonInfo(BaseModel):
    name: str = Field(description="人物姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")
    skills: list[str] = Field(description="技能列表")

# 创建 JSON 解析器
json_parser = JsonOutputParser(pydantic_object=PersonInfo)

# 创建提示词，包含格式说明
json_prompt = ChatPromptTemplate.from_template(
    """从以下文本中提取人物信息：
{text}

{format_instructions}
"""
)

# 创建链
json_chain = (
    {
        "text": RunnablePassthrough(),
        "format_instructions": lambda _: json_parser.get_format_instructions()
    }
    | json_prompt
    | llm
    | json_parser
)

# 测试
result = json_chain.invoke(
    "张三今年28岁，是一名软件工程师，擅长Python、机器学习和数据分析。"
)

print("🔍 JSON 解析结果:\n")
print(f"类型: {type(result)}")
print(f"内容: {result}")

## 🚀 11. 完整实战案例 - 智能文章生成系统

### 案例说明

这个案例综合运用了前面学到的所有 LCEL 技术，构建一个完整的文章生成系统，包括：
- 主题扩展
- 大纲生成
- 并行生成多个章节
- 内容整合
- 最终审核

In [ ]:
# 步骤1: 主题扩展
topic_expansion_chain = (
    ChatPromptTemplate.from_template(
        "将主题'{topic}'扩展为一个详细的写作主题，包括3个要点"
    )
    | llm
    | StrOutputParser()
)

# 步骤2: 生成大纲
outline_chain = (
    ChatPromptTemplate.from_template(
        "为以下主题生成文章大纲（3个章节）：\n{expanded_topic}"
    )
    | llm
    | StrOutputParser()
)

# 步骤3: 生成引言
intro_chain = (
    ChatPromptTemplate.from_template(
        "为以下主题写一段引人入胜的引言（50字以内）：\n{expanded_topic}"
    )
    | llm
    | StrOutputParser()
)

# 步骤4: 生成结论
conclusion_chain = (
    ChatPromptTemplate.from_template(
        "为以下主题写一段简洁的结论（50字以内）：\n{expanded_topic}"
    )
    | llm
    | StrOutputParser()
)

# 组合完整的文章生成流程
article_generation_chain = (
    # 首先扩展主题
    {"expanded_topic": topic_expansion_chain, "original_topic": lambda x: x["topic"]}
    # 然后并行生成大纲、引言和结论
    | RunnablePassthrough.assign(
        outline=outline_chain,
        introduction=intro_chain,
        conclusion=conclusion_chain
    )
    # 最后格式化输出
    | RunnableLambda(lambda x: f"""
{'='*60}
文章主题: {x['original_topic']}
{'='*60}

【扩展主题】
{x['expanded_topic']}

{'='*60}
【文章大纲】
{'='*60}
{x['outline']}

{'='*60}
【引言】
{'='*60}
{x['introduction']}

{'='*60}
【结论】
{'='*60}
{x['conclusion']}
{'='*60}
""")
)

# 执行完整流程
print("🚀 智能文章生成系统启动...\n")
final_article = article_generation_chain.invoke({"topic": "人工智能的未来"})
print(final_article)

## 🌊 12. 实战案例的流式版本

### 案例说明

将上面的文章生成系统改造为流式输出版本，提供更好的用户体验。

In [ ]:
# 创建简化的流式文章生成链
streaming_article_chain = (
    ChatPromptTemplate.from_template(
        """请为主题'{topic}'写一篇短文，包括：
1. 引言
2. 主要内容（2-3个要点）
3. 结论

文章总长度控制在200字以内。
"""
    )
    | llm
    | StrOutputParser()
)

# 流式输出
print("🌊 流式生成文章:\n")
print("="*60)
for chunk in streaming_article_chain.stream({"topic": "云计算技术"}):
    print(chunk, end="", flush=True)
print("\n" + "="*60)
print("\n✅ 文章生成完成！")

## 📊 13. 批量处理实战 - 多主题文章生成

### 案例说明

使用批处理同时为多个主题生成摘要。

In [ ]:
# 创建摘要生成链
summary_generation_chain = (
    ChatPromptTemplate.from_template(
        "用一句话概括'{topic}'的核心价值"
    )
    | llm
    | StrOutputParser()
)

# 准备多个主题
topics = [
    {"topic": "区块链"},
    {"topic": "物联网"},
    {"topic": "5G技术"},
    {"topic": "边缘计算"},
    {"topic": "量子计算"},
]

# 批量处理
print("📊 批量生成主题摘要:\n")
results = summary_generation_chain.batch(topics)

for topic, summary in zip(topics, results):
    print(f"🔹 {topic['topic']:8s}: {summary}")

## 🎓 14. 总结和最佳实践

### LCEL 核心概念回顾

1. **管道操作符 `|`**: 连接 Runnable 组件
2. **Runnable 接口**: 统一的 invoke/batch/stream 方法
3. **RunnableParallel**: 并行执行多个链
4. **RunnableBranch**: 条件分支
5. **RunnableLambda**: 自定义函数
6. **RunnablePassthrough**: 数据传递和上下文保持

### 最佳实践

✅ **推荐做法**:
- 使用 LCEL 构建可复用的链
- 利用并行处理提高效率
- 使用流式输出改善用户体验
- 用 RunnablePassthrough 保持上下文
- 使用 Output Parser 结构化输出

❌ **避免做法**:
- 过度复杂的链嵌套
- 忽略错误处理
- 在不需要时使用并行处理（增加复杂度）

### 调试技巧

```python
# 1. 打印中间结果
chain = (
    prompt 
    | RunnableLambda(lambda x: print(f"Debug: {x}") or x)
    | llm
)

# 2. 测试链的每个部分
result1 = prompt.invoke({"input": "test"})
result2 = llm.invoke(result1)

# 3. 使用 batch 测试多个案例
test_cases = [{"input": "test1"}, {"input": "test2"}]
results = chain.batch(test_cases)
```

## 🎯 15. 练习题

### 练习 1: 创建一个多语言翻译链

要求:
- 输入一段文本
- 并行翻译成英语、日语、韩语
- 返回包含三种翻译的字典

In [ ]:
# 在这里完成练习 1
# 提示: 使用 RunnableParallel 和多个翻译链

# your code here
pass

### 练习 2: 创建条件路由链

要求:
- 输入一个问题
- 如果问题包含"代码"，使用技术风格回答
- 如果问题包含"故事"，使用文学风格回答
- 其他情况使用通用风格回答

In [ ]:
# 在这里完成练习 2
# 提示: 使用 RunnableBranch

# your code here
pass

### 练习 3: 创建流式对话系统

要求:
- 创建一个简单的对话链
- 支持流式输出
- 能够保留历史对话上下文

In [ ]:
# 在这里完成练习 3
# 提示: 使用 RunnablePassthrough 保存历史，stream 进行流式输出

# your code here
pass

## 📚 参考资源

- [LangChain 官方文档](https://python.langchain.com/docs/expression_language/)
- [LangChain 中文文档](https://python.langchain.com.cn/docs/expression_language/)
- [LCEL GitHub 示例](https://github.com/langchain-ai/langchain)

## 🙏 结语

恭喜你完成了 LCEL 学习 Demo！通过本教程，你应该已经掌握了：

✅ LCEL 的基本概念和语法  
✅ 如何使用管道操作符构建链  
✅ 流式处理、批处理和并行处理  
✅ 条件分支和自定义函数  
✅ 数据传递和上下文管理  
✅ 构建完整的实战应用  

继续探索和实践，你将能够构建更强大的 LangChain 应用！🚀